# Four-pixel autoregressive model

Make chain-rule factorisation, exact likelihood evaluation, and sequential stochastic sampling executable.

**Classroom pattern:** predict first, run one cell, inspect the intermediate state, then change one parameter. All random seeds are fixed for the first run.

## Before you run

- Use the environment from `README.md`; no lecturer-specific Conda environment is required.
- Set `LECTURE2_QUICK=1` before starting Jupyter for the CPU-safe verification path.
- CUDA is selected first, then Apple MPS, then CPU. A separate MLX installation is not required.
- Read the prediction prompt before each experiment. The fixed-seed result is a mechanism check, not a quality benchmark.
- Every generated file stays inside `fallbacks/` or `checkpoints/`. No cell uploads or publishes anything.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "demo_models.py").exists():
    candidate = ROOT / "lecture 2" / "demos"
    if candidate.exists():
        ROOT = candidate
sys.path.insert(0, str(ROOT))
(ROOT / "fallbacks").mkdir(exist_ok=True)
print(f"Demo root: {ROOT}")

import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)

## 1. One conditional distribution per position

The functions below return the probability that the next binary pixel equals 1. The prefix determines the conditional.

In [ ]:
def p_one(prefix):
    if len(prefix) == 0:
        return 0.80
    if len(prefix) == 1:
        return 0.25 if prefix[0] == 1 else 0.55
    if len(prefix) == 2:
        return 0.70 if prefix == [1, 0] else 0.40
    if len(prefix) == 3:
        table = {(1, 0, 1): 0.90, (1, 1, 1): 0.45, (1, 0, 0): 0.30}
        return table.get(tuple(prefix), 0.50)
    raise ValueError("This demo contains four pixels")

def probability_of_value(value, prefix):
    p1 = p_one(prefix)
    return p1 if value == 1 else 1.0 - p1

## 2. Exact likelihood for a fixed observed image

Predict the product before running the cell. Exact means that every normalised factor can be evaluated without Monte Carlo approximation.

In [ ]:
observed = [1, 0, 1, 1]
factors = [probability_of_value(value, observed[:i]) for i, value in enumerate(observed)]
joint = float(np.prod(factors))
nll = -float(np.log(joint))
print("observed image:", observed)
print("factors:       ", factors)
print(f"p(x) = {' × '.join(f'{p:.2f}' for p in factors)} = {joint:.3f}")
print(f"negative log-likelihood = {nll:.3f}")

## 3. Sequential sampling

At generation time the model samples one pixel, appends it to the context, and asks the next conditional question.

In [ ]:
def sample_image(seed):
    rng = np.random.default_rng(seed)
    prefix, trace = [], []
    for position in range(4):
        p1 = p_one(prefix)
        value = int(rng.random() < p1)
        prefix.append(value)
        trace.append({"position": position + 1, "p_one": p1, "sample": value, "state": prefix.copy()})
    return prefix, trace

sample_a, trace_a = sample_image(8)    # seed A on the slide -> [1, 0, 1, 1]
sample_b, trace_b = sample_image(67)   # seed B on the slide -> [1, 1, 0, 1]
print("seed A (8):", sample_a)
print("seed B (67):", sample_b)
for row in trace_a:
    print(row)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 2.2))
axes[0].imshow(np.full((2, 2), np.nan), vmin=0, vmax=1, cmap="Blues")
axes[0].set_title("start")
for ax, row in zip(axes[1:], trace_a):
    canvas = np.full(4, np.nan)
    canvas[:len(row["state"])] = row["state"]
    ax.imshow(canvas.reshape(2, 2), vmin=0, vmax=1, cmap="Blues")
    ax.set_title(f"sample x{row['position']}\np(1)={row['p_one']:.2f}")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("One random draw extends the observed prefix")
fig.tight_layout()
fig.savefig(ROOT / "fallbacks" / "01_autoregressive_sampling.png", dpi=180, bbox_inches="tight")
plt.show()

## 4. Intervention: change the second pixel

The first two factors remain local to their positions, while later conditionals can change because their context changed.

In [ ]:
for prefix in ([1, 0], [1, 1]):
    print(f"prefix {prefix}: p(x3=1 | prefix)={p_one(list(prefix)):.2f}")
print("This is dependency through the conditional context, not a change to the chain rule itself.")

## Interpretation check and optional extension

Explain why exact likelihood evaluation does not make sampling deterministic. Then change one conditional probability and identify exactly which observed-image factors and sampled prefixes can change.

**Optional extension:** replace the hand-written table with a tiny masked neural network and compare its learned conditionals with the table.

Record the changed parameter, the fixed controls, the observed evidence, and one limitation.